# 14 — Out-of-distribution test: generators the system has never seen

Tests the **final hybrid system** (same code as `app.py`) on AI images from sources that are **not** in the training data:

| Part | Images | Expected answer |
|---|---|---|
| A | **COCO_AI** (Hugging Face): one real COCO photo + the same caption drawn by **SD 3.5, SD 3, SD 2.1, SDXL, DALL·E 3, Midjourney v6** | Real / Full AI |
| B | **Stable Diffusion 1.5 inpainting** applied to the untouched originals of **test** photos, inside the **same verified masks** as their Flux edits | Partial AI, with the mask as the answer key |

Part B compares the two generators on identical photos and masks, so any difference comes from the generator alone.

Needs internet the first time: COCO_AI rows are streamed (only what is used, ~100 MB) and the inpainting model is ~2 GB.
Run on the laptop GPU; close the app and other notebooks first. Everything is saved under `ood_test/` and `results/ood/`.

In [1]:
# ============================================================
# 1. Setup: load the final system exactly as the app does
# ============================================================
import os, sys, subprocess, importlib, json, warnings, random
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
PROJECT_DIR = (Path("/content/drive/MyDrive/AI_Image_Project") if IN_COLAB
               else Path(os.environ.get("AI_PROJECT_DIR", r"C:\AI_Image_Project")))   # <- laptop folder
os.environ["AI_PROJECT_DIR"] = str(PROJECT_DIR)

def ensure(module, pip_name=None):
    try: __import__(module)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])
for m, p in [("gradio", None), ("datasets", None), ("diffusers", None), ("accelerate", None), ("transformers", None)]:
    ensure(m, p)

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings("ignore"); pd.set_option("display.width", 200)

assert (PROJECT_DIR / "app.py").exists(), f"Put app.py in {PROJECT_DIR}"
sys.path.insert(0, str(PROJECT_DIR))
app = importlib.import_module("app")      # loads both U-Nets + the hybrid decision model (same code as the UI)
CLASS_NAMES, MASK_T = app.CLASS_NAMES, app.MASK_T
OOD = PROJECT_DIR / "ood_test"; OUT = PROJECT_DIR / "results" / "ood"
OOD.mkdir(exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

N_ROWS = 50          # COCO_AI rows -> up to 50 real + 6 x 50 AI images
N_INPAINT = 60       # test photos re-edited with Stable Diffusion 1.5 inpainting
SEED = 0

def run(img):
    arr, hu, h7, probs, cls, area = app.run_models(app.prepare(img))
    return arr, h7, probs, cls, area
print("Final system ready on", app.DEVICE)

Models loaded on cuda: unified_Unet_resnet34_crop384 + Unet_resnet34_crop512_plusbbox_clean | mask threshold 0.25, area threshold 0.7
Final system ready on cuda


In [2]:
# ============================================================
# 2. Part A data: stream COCO_AI rows (real photo + 6 generators for the same caption)
# ============================================================
from datasets import load_dataset
COLS = {"coco_image": "real_coco", "sd35_image": "sd3.5", "sd3_image": "sd3", "sd21_image": "sd2.1",
        "sdxl_image": "sdxl", "dalle_image": "dalle3", "midjourney_image": "midjourney6"}

def ahash(img, n=16):                       # small perceptual hash to find duplicates of our own COCO photos
    a = np.asarray(img.convert("L").resize((n, n), Image.Resampling.BILINEAR), np.float32)
    return (a > a.mean()).flatten()

coco_hashes = None
coco_dir = PROJECT_DIR / "COCO_5000"
if coco_dir.exists():
    hs = []
    for f in tqdm(sorted(coco_dir.glob("*.jpg")), desc="Hashing our COCO_5000 (duplicate check)"):
        try: hs.append(ahash(Image.open(f)))
        except Exception: pass                 # unreadable file: ignore
    coco_hashes = np.stack(hs) if hs else None

partA = OOD / "partA"
if not (partA / "index.csv").exists():
    rows, kept = [], 0
    stream = load_dataset("NasrinImp/COCO_AI", split="train", streaming=True)
    for i, r in enumerate(tqdm(stream, total=N_ROWS * 2, desc="Streaming COCO_AI")):
        if kept >= N_ROWS: break
        real = r.get("coco_image")
        if real is None: continue
        if coco_hashes is not None and (np.abs(coco_hashes ^ ahash(real)[None]).sum(1) <= 12).any():
            continue                          # this real photo is (nearly) one of our training photos -> skip the row
        for col, name in COLS.items():
            im = r.get(col)
            if im is None: continue
            d = partA / name; d.mkdir(parents=True, exist_ok=True)
            p = d / f"{kept:03d}.png"; im.convert("RGB").save(p)
            rows.append({"path": str(p), "source": name, "expected": 0 if name == "real_coco" else 1})
        kept += 1
    pd.DataFrame(rows).to_csv(partA / "index.csv", index=False)
idxA = pd.read_csv(partA / "index.csv")
print(idxA.groupby("source").size().to_string())

Hashing our COCO_5000 (duplicate check):   0%|          | 0/5000 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/3.15k [00:00<?, ?B/s]

Streaming COCO_AI:   0%|          | 0/100 [00:00<?, ?it/s]

source
dalle3         50
midjourney6    50
real_coco      50
sd2.1          50
sd3            50
sd3.5          50
sdxl           50


In [3]:
# ============================================================
# 3. Part B data: Stable Diffusion inpainting (1.5, else 2) on TEST originals, inside the same verified masks as the Flux edits
# ============================================================
BASE = PROJECT_DIR / "Partial_AI_5000"
pairs = pd.read_csv(BASE / "pairs.csv"); pairs["row"] = np.arange(len(pairs))
audit = pd.read_csv(PROJECT_DIR / "features" / "localization" / "mask_audit.csv").set_index("row")
segm = pairs[pairs.mask_type == "segm"].reset_index(drop=True)
test_sources = set(segm.source_id.to_numpy()[np.load(PROJECT_DIR / "features" / "localization" / "test_idx.npy")])
cand = pairs[(pairs.mask_type == "segm") & pairs.source_id.isin(test_sources)].copy()
cand = cand[audit.loc[cand.row, "verdict"].to_numpy() == "ok"]          # verified mask, original same size
cand = cand.drop_duplicates("source_id").sample(frac=1, random_state=SEED).head(N_INPAINT)
print("Test photos used for Part B:", len(cand))

partB = OOD / "partB"; partB.mkdir(exist_ok=True)
if not (partB / "index.csv").exists():
    from diffusers import StableDiffusionInpaintPipeline
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    pipe = None
    for repo in ["stable-diffusion-v1-5/stable-diffusion-inpainting", "runwayml/stable-diffusion-inpainting",
                 "stabilityai/stable-diffusion-2-inpainting"]:
        for kw in ({"variant": "fp16"}, {}):
            try:
                pipe = StableDiffusionInpaintPipeline.from_pretrained(repo, torch_dtype=dtype, safety_checker=None, **kw); break
            except Exception as e:
                print(f"  could not load {repo} {kw}: {str(e)[:120]}")
        if pipe is not None:
            print("Inpainting model:", repo); (partB / "model.txt").write_text(repo); break
    assert pipe is not None, "Could not download an SD inpainting model - check the internet connection"
    pipe = pipe.to(app.DEVICE); pipe.enable_attention_slicing(); pipe.set_progress_bar_config(disable=True)

    rows = []
    for k, r in enumerate(tqdm(cand.itertuples(), total=len(cand), desc="SD 1.5 inpainting")):
        orig = Image.open(BASE / r.original_path).convert("RGB")
        mask = Image.open(BASE / r.mask_path).convert("L")
        if mask.size != orig.size: mask = mask.resize(orig.size, Image.Resampling.NEAREST)
        W, H = orig.size; s = 512 / max(W, H); w8, h8 = max(64, int(W * s) // 8 * 8), max(64, int(H * s) // 8 * 8)
        g = torch.Generator(device=app.DEVICE).manual_seed(SEED + k)
        out = pipe(prompt="a photo", image=orig.resize((w8, h8), Image.Resampling.BICUBIC), mask_image=mask.resize((w8, h8), Image.Resampling.NEAREST),
                   num_inference_steps=25, guidance_scale=7.5, generator=g).images[0].resize((W, H), Image.Resampling.BICUBIC)
        m = np.asarray(mask) > 127
        edited = np.asarray(orig).copy(); edited[m] = np.asarray(out)[m]      # paste back: outside the mask stays untouched
        ep, mp = partB / f"sd15_{k:03d}.png", partB / f"mask_{k:03d}.png"
        Image.fromarray(edited).save(ep); mask.save(mp)
        rows.append({"k": k, "sd_edit": str(ep), "mask": str(mp), "flux_edit": str(BASE / r.manipulated_path),
                     "original": str(BASE / r.original_path), "true_area": 100 * m.mean()})
    pd.DataFrame(rows).to_csv(partB / "index.csv", index=False)
    del pipe; torch.cuda.empty_cache()
idxB = pd.read_csv(partB / "index.csv")
SD_NAME = {"stabilityai/stable-diffusion-2-inpainting": "SD 2"}.get((partB / "model.txt").read_text().strip(), "SD 1.5") \
          if (partB / "model.txt").exists() else "SD"
print("Part B images:", len(idxB), "| editor:", SD_NAME)

Test photos used for Part B: 60


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_inpaint.StableDiffusionInpaintPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Inpainting model: stable-diffusion-v1-5/stable-diffusion-inpainting


SD 1.5 inpainting:   0%|          | 0/60 [00:00<?, ?it/s]

Part B images: 60 | editor: SD 1.5


In [4]:
# ============================================================
# 4. Run the final system on everything and score it
# ============================================================
resA = []
for r in tqdm(idxA.itertuples(), total=len(idxA), desc="Part A"):
    _, _, probs, cls, area = run(Image.open(r.path))
    resA.append({"source": r.source, "expected": r.expected, "pred": cls, "conf": probs[cls], "area": area})
resA = pd.DataFrame(resA)

def f1(pred_mask, true_mask):
    i = (pred_mask & true_mask).sum(); return 2 * i / (pred_mask.sum() + true_mask.sum() + 1e-9)

resB = []
for r in tqdm(idxB.itertuples(), total=len(idxB), desc="Part B"):
    true = np.asarray(Image.open(r.mask).convert("L")) > 127
    for kind, path in [(f"{SD_NAME} edit", r.sd_edit), ("Flux edit (same photo, same mask)", r.flux_edit), ("untouched original", r.original)]:
        arr, h7, probs, cls, area = run(Image.open(path))
        t = true if true.shape == h7.shape else np.asarray(Image.fromarray(true).resize(h7.shape[::-1], Image.Resampling.NEAREST))
        exp = 0 if kind == "untouched original" else 2
        resB.append({"kind": kind, "expected": exp, "pred": cls, "conf": probs[cls], "area": area,
                     "true_area": 0.0 if exp == 0 else r.true_area,
                     "mask_f1": f1(h7 >= MASK_T, t) if exp == 2 else np.nan})
resB = pd.DataFrame(resB)

def summary(df, by):
    out = []
    for name, d in df.groupby(by, sort=False):
        out.append({by: name, "images": len(d), "correct %": 100 * (d.pred == d.expected).mean(),
                    "called Real %": 100 * (d.pred == 0).mean(), "called Full AI %": 100 * (d.pred == 1).mean(),
                    "called Partial AI %": 100 * (d.pred == 2).mean()})
    return pd.DataFrame(out)

print("=" * 80); print("PART A - real photos and FULLY AI images from unseen generators"); print("=" * 80)
sA = summary(resA, "source"); print(sA.round(1).to_string(index=False))
print("\n" + "=" * 80); print("PART B - PARTIAL AI: Stable Diffusion vs Flux on the same photos and masks"); print("=" * 80)
sB = summary(resB, "kind")
sB["mean mask F1"] = [resB[resB.kind == k].mask_f1.mean() for k in sB.kind]
sB["area error (pp)"] = [np.abs(resB[resB.kind == k].area - resB[resB.kind == k].true_area).mean() for k in sB.kind]
print(sB.round(3).to_string(index=False))
sA.to_csv(OUT / "partA_summary.csv", index=False); sB.to_csv(OUT / "partB_summary.csv", index=False)
resA.to_csv(OUT / "partA_per_image.csv", index=False); resB.to_csv(OUT / "partB_per_image.csv", index=False)

Part A:   0%|          | 0/350 [00:00<?, ?it/s]

Part B:   0%|          | 0/60 [00:00<?, ?it/s]

PART A - real photos and FULLY AI images from unseen generators
     source  images  correct %  called Real %  called Full AI %  called Partial AI %
  real_coco      50       92.0           92.0               2.0                  6.0
      sd3.5      50        4.0           62.0               4.0                 34.0
        sd3      50       20.0           30.0              20.0                 50.0
      sd2.1      50       46.0           36.0              46.0                 18.0
       sdxl      50       28.0           36.0              28.0                 36.0
     dalle3      50       14.0           82.0              14.0                  4.0
midjourney6      50       12.0           80.0              12.0                  8.0

PART B - PARTIAL AI: Stable Diffusion vs Flux on the same photos and masks
                             kind  images  correct %  called Real %  called Full AI %  called Partial AI %  mean mask F1  area error (pp)
                      SD 1.5 edit      60 

In [5]:
# ============================================================
# 5. Figures for the report
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
order = list(sA.source)
ax[0].barh(order, sA["correct %"], color=["#2ca02c" if s == "real_coco" else "#d62728" for s in order])
for i, v in enumerate(sA["correct %"]): ax[0].text(v + 1, i, f"{v:.0f}%", va="center")
ax[0].set_xlim(0, 105); ax[0].set_xlabel("% correct"); ax[0].set_title("Part A: unseen generators (Full AI) and real photos")
ax[1].barh(list(sB.kind), sB["correct %"], color=["#ff7f0e", "#1f77b4", "#2ca02c"][:len(sB)])
for i, v in enumerate(sB["correct %"]): ax[1].text(v + 1, i, f"{v:.0f}%", va="center")
ax[1].set_xlim(0, 105); ax[1].set_xlabel("% correct"); ax[1].set_title("Part B: same photos, same masks, different editor")
plt.tight_layout(); plt.savefig(OUT / "ood_summary.png", dpi=110); plt.show()

ex = idxB.head(3)
fig, ax = plt.subplots(len(ex), 5, figsize=(22, 4.2 * len(ex)), squeeze=False)
for i, r in enumerate(ex.itertuples()):
    o = np.asarray(Image.open(r.original).convert("RGB")); t = np.asarray(Image.open(r.mask).convert("L")) > 127
    for j, (path, title) in enumerate([(r.flux_edit, "Flux edit"), (r.sd_edit, f"{SD_NAME} edit")]):
        arr, h7, probs, cls, area = run(Image.open(path)); ov = arr.copy(); ov[h7 >= MASK_T] = (0.5 * ov[h7 >= MASK_T] + [127, 0, 0]).astype(np.uint8)
        ax[i, 2 + j].imshow(ov); ax[i, 2 + j].set_title(f"{title}: {CLASS_NAMES[cls]} ({probs[cls]*100:.0f}%), area {area:.1f}%")
    ax[i, 0].imshow(o); ax[i, 0].set_title("original")
    ax[i, 1].imshow(t, cmap="gray"); ax[i, 1].set_title(f"true edit mask ({100*t.mean():.1f}%)")
    ax[i, 4].imshow(np.asarray(Image.open(r.sd_edit).convert("RGB"))); ax[i, 4].set_title(f"{SD_NAME} edit (no overlay)")
    for a in ax[i]: a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "ood_partB_examples.png", dpi=90); plt.show()
print("Saved figures and tables in", OUT)

Saved figures and tables in C:\AI_Image_Project\results\ood
